In [1]:
import re
import os
import ast
import math
import shutil
import birankpy
import numpy as np
import pandas as pd
from PIL import Image
import networkx as nx
from scipy import stats
import word2image as w2i
import scipy.stats as stats
from tqdm import tqdm_notebook
from collections import Counter
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize
from nltk.corpus import stopwords, wordnet


# This part declares and utilizes variables and functions from other .py files.

## The reason for not calling them directly is to ensure that this .ipynb file can run independently, even without the respective .py files.

In [2]:
"""
These functions are used to extract the first, second, or third element from each sublist 
in a list of lists, where each sublist contains three elements.
"""

def leave_last_element(List):
    lst=[]    
    for x in List:
            lst.append(x[2])
    return lst

def leave_middle_element(List):
    lst=[]
    for x in List:
        lst.append(x[1])
    return lst

def leave_first_element(List):
    lst=[]
    for x in List:
        lst.append(x[0])
    return lsta

In [3]:
"""
To create a word-color association that maximally reflects human cognitive abilities, the LCH system has been adopted.
However, for the visualization process, conversion to RGB is necessary.
"""

def xyz_to_rgb(xyz):
    """
    Converts XYZ color space values to RGB color space.

    Args:
        xyz (list of tuples): List of tuples where each tuple contains X, Y, and Z values.

    Returns:
        list of tuples: List of tuples where each tuple contains R, G, and B values.
    """
    # Transformation matrix
    M = [[3.2406, -1.5372, -0.4986],
         [-0.9689, 1.8758, 0.0415],
         [0.0557, -0.2040, 1.0570]]
    
    rgb = []
    
    for x, y, z in xyz:
        # Normalize the values
        x /= 100
        y /= 100
        z /= 100
        
        # Convert XYZ to RGB
        r = M[0][0] * x + M[0][1] * y + M[0][2] * z
        g = M[1][0] * x + M[1][1] * y + M[1][2] * z
        b = M[2][0] * x + M[2][1] * y + M[2][2] * z
        
        # Apply gamma correction
        r = 1.055 * (r ** (1 / 2.4)) - 0.055 if r > 0.0031308 else r * 12.92
        g = 1.055 * (g ** (1 / 2.4)) - 0.055 if g > 0.0031308 else g * 12.92
        b = 1.055 * (b ** (1 / 2.4)) - 0.055 if b > 0.0031308 else b * 12.92
        
        # Clip values between 0 and 1
        r = max(0, min(1, r))
        g = max(0, min(1, g))
        b = max(0, min(1, b))
        
        rgb.append((r, g, b))
    
    return rgb

def lch_to_xyz(l, c, h):
    """
    Converts LCH color space values to XYZ color space.

    Args:
        l (float): Lightness value between 0 and 100.
        c (float): Chroma value between 0 and 100.
        h (float): Hue value between 0 and 360.

    Returns:
        tuple: Tuple of floats (X, Y, Z) representing the XYZ values.
    """
    # Convert hue to radians
    h_rad = math.radians(h)
    
    # Calculate the a and b values
    a = c * math.cos(h_rad)
    b = c * math.sin(h_rad)
    
    # Calculate the intermediate values
    fy = (l + 16.0) / 116.0
    fx = fy + (a / 500.0)
    fz = fy - (b / 200.0)
    
    # Calculate the XYZ values
    x_n = 0.95047  # D65 reference white
    y_n = 1.0
    z_n = 1.08883
    x = x_n * (fx ** 3 if fx ** 3 > 0.008856 else (fx - 16.0/116.0) / 7.787) * 100
    y = y_n * (fy ** 3 if fy ** 3 > 0.008856 else (fy - 16.0/116.0) / 7.787) * 100
    z = z_n * (fz ** 3 if fz ** 3 > 0.008856 else (fz - 16.0/116.0) / 7.787) * 100
    
    return x, y, z


In [4]:
def extractColors_before_update(resize_images_dir, search_term=""):
    """
    Extracts color information from an image file and converts it into various color spaces.

    Args:
        resize_images_dir (str): Base directory where the resized image files are located.
        search_term (str): Search term or filename (without extension) of the image.

    Returns:
        list: A list containing lists of LCH values.
    """
    filename = search_term + ".png"
    img_url = os.path.join(resize_images_dir, filename)
    
    # Open the image file
    image = Image.open(img_url)
    
    # Get the size of the image in pixels
    width, height = image.size

    # Create an empty list to store the RGB values for each pixel
    rgb_list = []
    lab_list = []

    # Loop over each pixel in the image and extract the RGB values
    for y in range(height):
        for x in range(width):
            pixel = image.getpixel((x, y))
            if isinstance(pixel, int):
                r, g, b = pixel, pixel, pixel
            else:
                r, g, b = pixel[:3]
            rgb_list.append((r, g, b))

    for (r, g, b) in rgb_list:
        if isinstance(r, int) and isinstance(g, int) and isinstance(b, int) and 0 <= r <= 255 and 0 <= g <= 255 and 0 <= b <= 255:
            # Convert RGB to XYZ
            r_norm = r / 255
            g_norm = g / 255
            b_norm = b / 255

            r_lin = r_norm if r_norm <= 0.04045 else ((r_norm + 0.055) / 1.055) ** 2.4
            g_lin = g_norm if g_norm <= 0.04045 else ((g_norm + 0.055) / 1.055) ** 2.4
            b_lin = b_norm if b_norm <= 0.04045 else ((b_norm + 0.055) / 1.055) ** 2.4

            x = r_lin * 0.4124564 + g_lin * 0.3575761 + b_lin * 0.1804375
            y = r_lin * 0.2126729 + g_lin * 0.7151522 + b_lin * 0.0721750
            z = r_lin * 0.0193339 + g_lin * 0.1191920 + b_lin * 0.9503041

            # Convert XYZ to Lab
            xn = 0.95047
            yn = 1.00000
            zn = 1.08883

            f_x = x ** (1/3) if x > 0.008856 else 7.787 * x + 16 / 116
            f_y = y ** (1/3) if y > 0.008856 else 7.787 * y + 16 / 116
            f_z = z ** (1/3) if z > 0.008856 else 7.787 * z + 16 / 116

            L = 116 * f_y - 16
            a = 500 * (f_x - f_y)
            b = 200 * (f_y - f_z)

            lab_list.append((L, a, b))
        else:
            print("Invalid RGB value:", (r, g, b))

    if not lab_list:
        return None

    # Convert CIELAB values to CIELCH values
    lch_list_before_update = []

    for lab in lab_list:
        L, a, b = lab
        C = np.sqrt(a**2 + b**2)
        H = np.degrees(np.arctan2(b, a))
        if H < 0:
            H += 360
        lch_list_before_update.append((L, C, H))

    return lch_list_before_update


In [5]:
#Proportion of lightness 100.0 within 100 random words
all_ratios= [0.36882398777615333, 0.338222491268681, 0.33556438840155944, 0.3298635223765432, 0.36380792113385313, 0.35463039108187133, 0.3505666473765432, 0.36405234730344377, 0.34570350572612085, 0.36602780823586745, 0.34529600186809617, 0.3420673174545159, 0.3384813850308642, 0.32599839079759585, 0.34010048631416506, 0.3519716536712151, 0.32777841232131255, 0.32826904138239116, 0.3340251126949318, 0.3491021208983106, 0.3231354572774529, 0.33945134827810264, 0.3719376929012346, 0.34331647985705005, 0.35404445358187137, 0.3552032569850552, 0.33358347039473685, 0.35345851608187134, 0.35158369375406107, 0.35653668676900585, 0.3221067352988954, 0.3390547585688759, 0.3417280904808317, 0.3343795687134503, 0.358395772417154, 0.34707208922189736, 0.3412572591780377, 0.3411649965480832, 0.3244751055880442, 0.35186416199642623, 0.34296456201267056, 0.35780755056042884, 0.3261258071393762, 0.37569952079272256, 0.34405204272254714, 0.3276677479288499, 0.37083409478557505, 0.32230560124269003, 0.3556959165854451, 0.32262883771929823, 0.34073452221410006, 0.3447343039311241, 0.35195883589181287, 0.32796737938596493, 0.3430964201591943, 0.31672402940220923, 0.3619983298814165, 0.33748299423326833, 0.34265795057667314, 0.33877289433073426, 0.3368844927712801, 0.3654506274366472, 0.3346963328460039, 0.3340977044753086, 0.32277186383203377, 0.36450947246588694, 0.35352856968810914, 0.3505238791423002, 0.36978240233105913, 0.35253170179499677, 0.33897772498375567, 0.3436908605425601, 0.3376168829191033, 0.3197802448830409, 0.33266109791260556, 0.3541915407732294, 0.33375949277128003, 0.34395876482293697, 0.33755761655295646, 0.33195967348927874, 0.3247612847222222, 0.3219671357212476, 0.36218095151072127, 0.3591152178768681, 0.3206005827647823, 0.35638972648635475, 0.3290665356968811, 0.34599463429987004, 0.32994778975795974, 0.349654935225796, 0.3742771280051982, 0.31018493136777125, 0.31426593465724495, 0.3482470100308642, 0.3624507594217024, 0.3627684119152047, 0.33789481298732943, 0.3681160910087719, 0.3359024731968811, 0.32038775686322285]

#Now, when calculating the lightness_values for all random words, we trim the top 0.34329116918453534 (wild_mean) of lightness to remove the white lattice.
wild_mean = 0.34329116918453534 # wild_std = 0.015052801243747974

In [6]:
def extractlightness(resize_images_dir, search_term=""):
    """
    Extracts color information from an image file and converts it into various color spaces.

    Args:
        resize_images_dir (str): Base directory where the resized image files are located.
        search_term (str): Search term or filename (without extension) of the image.

    Returns:
        Updated Lightness values considering the white lattice in the screenshot.
        The key process is restoring as much as was removed by considering the distribution of the remaining values.
    """
    filename = search_term + ".png"
    img_url = os.path.join(resize_images_dir, filename)
    
    # Open the image file
    image = Image.open(img_url)
    
    # Get the size of the image in pixels
    width, height = image.size

    # Create an empty list to store the RGB values for each pixel
    rgb_list = []
    lab_list = []

    # Loop over each pixel in the image and extract the RGB values
    for y in range(height):
        for x in range(width):
            pixel = image.getpixel((x, y))
            if isinstance(pixel, int):
                r, g, b = pixel, pixel, pixel
            else:
                r, g, b = pixel[:3]
            rgb_list.append((r, g, b))

    for (r, g, b) in rgb_list:
        if isinstance(r, int) and isinstance(g, int) and isinstance(b, int) and 0 <= r <= 255 and 0 <= g <= 255 and 0 <= b <= 255:
            # Convert RGB to XYZ
            r_norm = r / 255
            g_norm = g / 255
            b_norm = b / 255

            r_lin = r_norm if r_norm <= 0.04045 else ((r_norm + 0.055) / 1.055) ** 2.4
            g_lin = g_norm if g_norm <= 0.04045 else ((g_norm + 0.055) / 1.055) ** 2.4
            b_lin = b_norm if b_norm <= 0.04045 else ((b_norm + 0.055) / 1.055) ** 2.4

            x = r_lin * 0.4124564 + g_lin * 0.3575761 + b_lin * 0.1804375
            y = r_lin * 0.2126729 + g_lin * 0.7151522 + b_lin * 0.0721750
            z = r_lin * 0.0193339 + g_lin * 0.1191920 + b_lin * 0.9503041

            # Convert XYZ to Lab
            xn = 0.95047
            yn = 1.00000
            zn = 1.08883

            f_x = x ** (1/3) if x > 0.008856 else 7.787 * x + 16 / 116
            f_y = y ** (1/3) if y > 0.008856 else 7.787 * y + 16 / 116
            f_z = z ** (1/3) if z > 0.008856 else 7.787 * z + 16 / 116

            L = 116 * f_y - 16
            a = 500 * (f_x - f_y)
            b = 200 * (f_y - f_z)

            lab_list.append((L, a, b))
        else:
            print("Invalid RGB value:", (r, g, b))

    if not lab_list:
        return None


    # Extract L values and compute the threshold to cut off the top 34.329116918453534%
    l_values = [lab[0] for lab in lab_list]
    threshold = np.percentile(l_values, 100 - (wild_mean*100.0))
    l_values_cut = [l for l in l_values if l <= threshold]

    # Create a histogram of the remaining L values
    l_hist, bins = np.histogram(l_values_cut, bins=np.arange(0, 101, 1))

    # Determine how many values were cut
    num_cut = len(l_values) - len(l_values_cut)

    # Distribute the cut values according to the histogram distribution
    l_distribution = l_hist / l_hist.sum()
    additional_values = np.random.choice(bins[:-1], size=num_cut, p=l_distribution)

    # Combine and sort the new L values
    new_l_values = np.sort(np.concatenate((l_values_cut, additional_values)))

    return new_l_values


In [7]:
red_word_list=["red","scarlet","vermilion","ruby","carmine"]
orange_word_list= ["orange","tangerine","marmalade","orangish","apricot"]
yellow_word_list = ["yellow","yellowish","yellowy","gold","golden"]
green_word_list = ["green","greenish","verdant","leafy","greenery"]
blue_word_list = ["blue","azure","cobalt","cerulean","ultramarine"]
purple_word_list = ["purple","violet","purply","purplish","amethyst"]
pink_word_list = ["pink","rosy","blushing","shellpink","rose"]
black_word_list = ["black","pitchblack","pitchdark","jetblack","blackish"]
grey_word_list = ["grey","silver","slategray","smokegray","silvery"]
white_word_list=["white","snowywhite","milkwhite","milkywhite","chalkwhite"]

In [8]:
color_nouns_list = [
    red_word_list,
    blue_word_list,
    green_word_list,
    yellow_word_list,
    purple_word_list,
    orange_word_list,
    black_word_list,
    white_word_list,
    grey_word_list,
    pink_word_list
]

color_nouns_list_1d=np.concatenate(color_nouns_list).tolist()

In [9]:
chroma_threshold = 15.930582082686795

In [10]:
def extracthue(resize_images_dir, search_term=""):
    """
    Extracts color information from an image file and converts it into various color spaces.

    Args:
        resize_images_dir (str): Base directory where the image files are located.
        search_term (str): Search term or filename (without extension) of the image.

    Returns:
        Returns:
        Updated Hue values considering the threshold.
        The key process is restoring as much as was removed by considering the distribution of the remaining values.
    """
    filename = search_term + ".png"
    img_url = os.path.join(resize_images_dir, filename)
    
    # Open the image file
    image = Image.open(img_url)
    
    # Get the size of the image in pixels
    width, height = image.size

    # Create an empty list to store the RGB values for each pixel
    rgb_list = []
    lab_list = []

    # Loop over each pixel in the image and extract the RGB values
    for y in range(height):
        for x in range(width):
            pixel = image.getpixel((x, y))
            if isinstance(pixel, int):
                r, g, b = pixel, pixel, pixel
            else:
                r, g, b = pixel[:3]
            rgb_list.append((r, g, b))

    for (r, g, b) in rgb_list:
        if isinstance(r, int) and isinstance(g, int) and isinstance(b, int) and 0 <= r <= 255 and 0 <= g <= 255 and 0 <= b <= 255:
            # Convert RGB to XYZ
            r_norm = r / 255
            g_norm = g / 255
            b_norm = b / 255

            r_lin = r_norm if r_norm <= 0.04045 else ((r_norm + 0.055) / 1.055) ** 2.4
            g_lin = g_norm if g_norm <= 0.04045 else ((g_norm + 0.055) / 1.055) ** 2.4
            b_lin = b_norm if b_norm <= 0.04045 else ((b_norm + 0.055) / 1.055) ** 2.4

            x = r_lin * 0.4124564 + g_lin * 0.3575761 + b_lin * 0.1804375
            y = r_lin * 0.2126729 + g_lin * 0.7151522 + b_lin * 0.0721750
            z = r_lin * 0.0193339 + g_lin * 0.1191920 + b_lin * 0.9503041

            # Convert XYZ to Lab
            xn = 0.95047
            yn = 1.00000
            zn = 1.08883

            f_x = x ** (1/3) if x > 0.008856 else 7.787 * x + 16 / 116
            f_y = y ** (1/3) if y > 0.008856 else 7.787 * y + 16 / 116
            f_z = z ** (1/3) if z > 0.008856 else 7.787 * z + 16 / 116

            L = 116 * f_y - 16
            a = 500 * (f_x - f_y)
            b = 200 * (f_y - f_z)

            lab_list.append((L, a, b))
        else:
            print("Invalid RGB value:", (r, g, b))

    if not lab_list:
        return None

    # Convert CIELAB values to CIELCH values
    lch_list = []

    for lab in lab_list:
        L, a, b = lab
        C = np.sqrt(a**2 + b**2)
        if C < chroma_threshold:
            H = -1  # Set H to -1 if chroma is too small to accurately determine hue
        else:
            H = np.degrees(np.arctan2(b, a))
            if H < 0:
                H += 360
        lch_list.append((L, C, H))

    # Extract H values
    h_values = leave_last_element(lch_list)
    h_minus_one_count = h_values.count(-1)

    # Calculate distribution of H values
    bins = np.linspace(0, 360, 361)
    h_distribution, _ = np.histogram([h for h in h_values if h != -1], bins=bins)

    # Calculate the ratio for each H value
    total_valid_h = sum(h_distribution)
    h_distribution_ratios = h_distribution / total_valid_h if total_valid_h > 0 else np.zeros(len(h_distribution))

    # Distribute the -1 H values
    distributed_counts = np.round(h_distribution_ratios * h_minus_one_count).astype(int)

    # Create new H value list
    new_h_values = [h for h in h_values if h != -1]
    for i, count in enumerate(distributed_counts):
        new_h_values.extend([i + 0.5] * count)

    return new_h_values


In [ ]:
# Define color ranges
color_ranges = {
    "red": [(19.326959847036328, 45.67667984189723)],
    "orange": [(45.67667984189723, 80.90534979423869)],
    "yellow": [(80.90534979423869, 120.1811320754717)],
    "green": [(120.1811320754717, 203.28)],
    "blue": [(203.28, 293.86897590361446)],
    "purple": [(293.86897590361446, 331.2451923076923)],
    "pink": [(331.2451923076923, 360), (0, 19.326959847036328)]
}

a_color_ranges = {
    "black": [(0, 41.65384615384615)],
    'grey': [(41.65384615384615, 82.89746682750301)],
    'white': [(82.89746682750301, 100)]
}

def transform_value(color, value):
    return stats.yeojohnson(value, lmbda=lambda_values[color])

def ratio_for_each_term(search_term):
    # Example list
    values = extracthue("/Users/wangseongpil/Desktop/Color Final/images/resize_images", search_term)

    # Initialize color counts
    color_counts = {color: 0 for color in color_ranges}

    # Determine which color range each value falls into and count
    for value in values:
        for color, ranges in color_ranges.items():
            for start, end in ranges:
                if start <= value % 360 <= end:
                    color_counts[color] += 1
                    break
    # Total number of values
    total_values = len(values)

    # Calculate and print the ratio for each color
    color_ratios = {color: count / total_values for color, count in color_counts.items()}

    print(color_ratios)
    
    a_values = extractlightness("/Users/wangseongpil/Desktop/Color Final/images/resize_images", search_term)

    # Initialize a color counts
    a_color_counts = {color: 0 for color in a_color_ranges}

    # Determine which color range each value falls into and count
    for value in a_values:
        for color, ranges in a_color_ranges.items():
            for start, end in ranges:
                if start <= value % 100 <= end:
                    a_color_counts[color] += 1
                    break
    # Total number of values
    a_total_values = len(a_values)

    # Calculate and print the ratio for each color
    color_ratios_a = {color: count / a_total_values for color, count in a_color_counts.items()}
    
    merged = {**color_ratios, **color_ratios_a}
    return merged

# "Comparison two poets"

### Code that allocates color composition to each word in the poet's work to compare.

### (1) Import CSV Files and Data Preprocessing

#### Books were collected and manually organized in the form of pdf files from sources specified in the paper.

#### Column Name : movement', 'writer', 'title', 'text', 'nouns', 'adjectives', 'verbs', 'adverbs'

##### - movement: This column indicates the literary movement to which the writer or work belongs. It reflects the stylistic and thematic characteristics of a particular era or school of thought in literature.

##### - writer: This column specifies the author of the text. It identifies the person responsible for writing the piece under analysis.

##### - book: This column contains the title of the book or collection from which the text is taken. It provides context about the larger body of work to which the text belongs.

##### - title: This column represents the title of the poem within the book. It helps identify the specific section or piece being analyzed.

##### - text: The text column contains the full excerpt of the passage or poem being studied. It serves as the primary source for analysis, showcasing the language and style of the writer.

##### - nouns: This column lists the key nouns present in the text. 

##### -  adjectives: This column lists the adjectives found within the text. 

##### - verbs: The verbs column highlights the action words in the text. 

##### - adverbs: This column includes the adverbs used in the text. 


#### This process loads a dataset and cleans the text by removing special characters and stopwords. It then tokenizes the text and categorizes words into nouns, verbs, adjectives, and adverbs using part-of-speech tagging and lemmatization. The result is a DataFrame with new columns for each word category based on the processed text.

In [12]:

# Function to convert POS tags to WordNet format
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return None

def load_and_process_data(file_path):
    # Load DataFrame
    df = pd.read_csv(file_path)
    print(df.columns)
    if 'Unnamed: 0' in df.columns:
        df = df.drop('Unnamed: 0', axis=1)
    if 'book' in df.columns:
        df = df.drop('book', axis=1)

    # Remove newline characters
    df.replace(to_replace=r'\n', value=' ', regex=True, inplace=True)

    # Keep only the first four columns
    df = df.iloc[:, :4]

    def categorize_words(text):
        # Remove extra spaces
        text = re.sub(r'\s+', ' ', text.strip())
        # Remove special characters and numbers
        text = re.sub(r'[^a-zA-Z ]', '', text)

        stop_words = set(stopwords.words('english'))
        lemmatizer = WordNetLemmatizer()

        words = word_tokenize(text)
        words = [word for word in words if word.isalpha() and word.lower() not in stop_words]
        tagged = pos_tag(words)

        nouns = []
        verbs = []
        adjectives = []
        adverbs = []

        for word, pos in tagged:
            wordnet_pos = get_wordnet_pos(pos)
            if wordnet_pos:
                lemmatized_word = lemmatizer.lemmatize(word, pos=wordnet_pos)
                if wordnet_pos == wordnet.NOUN:
                    nouns.append(lemmatized_word.lower())
                elif wordnet_pos == wordnet.VERB:
                    verbs.append(lemmatized_word.lower())
                elif wordnet_pos == wordnet.ADJ:
                    adjectives.append(lemmatized_word.lower())
                elif wordnet_pos == wordnet.ADV:
                    adverbs.append(lemmatized_word.lower())

        return nouns, verbs, adjectives, adverbs

    # Apply the function to the 'text' column
    df['nouns'], df['verbs'], df['adjectives'], df['adverbs'] = zip(*df['text'].apply(categorize_words))
    
    return df


In [14]:
#Amy Lowell
file_path = './csv/Amy_Lowell.csv'
df_im_al = load_and_process_data(file_path)

#Georgia Douglas Johnson
file_path = './csv/Georgia_Douglas_Johnson.csv'
df_hl_gdj = load_and_process_data(file_path)

In [ ]:
def merge_content_words(df):
    # Create the 'content_words' column by merging the lists from 'nouns', 'verbs', 'adjectives', and 'adverbs' columns
    df['content_words'] = df.apply(lambda x: x['nouns'] + x['verbs'] + x['adjectives'] + x['adverbs'], axis=1)
    
    # Remove the original four columns
    df.drop(columns=['nouns','verbs', 'adjectives', 'adverbs'], inplace=True)
    
    return df

In [ ]:
al_df_final=merge_content_words(df_im_al)
gdj_df_final=merge_content_words(df_hl_gdj)

##### word to original screenshot image, and resize

In [ ]:
original_dir = "./Images/original_images"
temp_dir = "./Images/temp"
resize_dir = "./Images/resize_images"

for word in tqdm_notebook(final_unique_words_list):
    image_filename = f"{word}.png"
    base_image_path = os.path.join(base_dir, image_filename)
    resize_image_path = os.path.join(resize_dir, image_filename)
    
    # word2image funtion call
    w2i.word2image(word, 10000, original_dir, temp_dir, resize_dir,check_text=True)

### (2) Assigning colors to each poetic word, with ratios assigned to the allocation.

In [ ]:
def apply_ratio_to_content_words(df):
    # New column to store the results
    df['raw_ratio_content_words'] = None
    
    # Ratios cache to avoid recalculating for the same terms
    ratios_cache = {}
    
    for index, row in tqdm_notebook(df.iterrows()):
        # Initialize a dictionary to hold the ratios for each term
        term_ratios = {}
        
        for term in row['content_words']:
            if term in ratios_cache:
                # Use cached ratio if available
                term_ratios[term] = ratios_cache[term]
            else:
                try:
                    # Attempt to compute ratio for each term and store it in the cache and dictionary
                    term_ratio = ratio_for_each_term(term)
                    ratios_cache[term] = term_ratio
                    term_ratios[term] = term_ratio
                except Exception as e:
                    # If an error occurs, skip this term
                    print(f"Skipping term '{term}' due to error: {e}")
                    continue
        
        # Assign the dictionary to the corresponding cell in 'raw_ratio_content_words'
        df.at[index, 'raw_ratio_content_words'] = term_ratios
    
    return df

In [ ]:
al_df_final_all=apply_ratio_to_content_words(al_df_final)
al_df_final_all.to_csv("./csv/processed_csv/al_df_with_ratios.csv")

gdj_df_final_all=apply_ratio_to_content_words(gdj_df_final)
gdj_df_final_all.to_csv("./csv/processed_csv/gdj_df_with_ratios.csv")

## (3) Performing bootstrap resampling of word–color associations.
#### Words are resampled according to their frequency distribution, and their associated color ratios are aggregated in each iteration. The process yields empirical distributions of color composition and evenness, from which the mean, standard deviation, and confidence intervals can be computed for each color.

In [ ]:
# -*- coding: utf-8 -*-
import os, ast
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from numpy.random import default_rng
from typing import Optional

# =========================
# User settings
# =========================
BASE_PATH = "./csv"

# Directory to save result CSVs
RESULTS_PATH = os.path.join(BASE_PATH, "results")
os.makedirs(RESULTS_PATH, exist_ok=True)

# Existing input files (preprocessed)
AL_FILE = os.path.join(BASE_PATH, "processed_csv", "al_df_with_ratios.csv")
GDJ_FILE = os.path.join(BASE_PATH, "processed_csv", "gdj_df_with_ratios.csv")

# Fixed color ordering
COLORS = ["red","orange","yellow","green","blue","purple","pink","black","grey","white"]
K = len(COLORS)

# =========================
# Utilities
# =========================
def safe_eval(x):
    """Safely parse lists/dicts stored as strings in CSV cells."""
    if isinstance(x, (list, dict)):
        return x
    if pd.isna(x):
        return []
    try:
        return ast.literal_eval(str(x))
    except Exception:
        return []

def merge_ratio_dicts(ratio_series: pd.Series) -> dict:
    """
    Merge per-row dictionaries of ratios into a global mapping:
    word -> {color: mean_ratio_over_rows}.
    """
    acc = defaultdict(lambda: defaultdict(list))
    for rmap in ratio_series:
        if not isinstance(rmap, dict):
            continue
        for w, cmap in rmap.items():
            if not isinstance(cmap, dict):
                continue
            for c, r in cmap.items():
                acc[w][c].append(float(r))
    merged = {}
    for w, cmap in acc.items():
        merged[w] = {c: float(np.mean(v)) for c, v in cmap.items()}
    return merged

def build_bow(df: pd.DataFrame) -> Counter:
    """Build a bag-of-words (token counts) from the 'content_words' column."""
    counts = Counter()
    for lst in df["content_words"]:
        if not isinstance(lst, list):
            continue
        counts.update(lst)
    return counts

def compute_color_scores_from_bow_ratio(
    bow_counts: Counter,
    ratio_master: dict,
) -> np.ndarray:
    """
    Compute the color score vector S (length K) by summing
    frequency-weighted ratios (Ratio Mode).
    """
    S = np.zeros(len(COLORS), dtype=float)
    for w, cnt in bow_counts.items():
        rmap = ratio_master.get(w)
        if not rmap:
            continue
        for j, color in enumerate(COLORS):
            R = float(rmap.get(color, 0.0))
            S[j] += cnt * R
    return S

def benjamini_hochberg(pvals):
    """Benjamini–Hochberg FDR correction. Returns a list of q-values."""
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    q = np.empty(m, dtype=float)
    prev = 1.0
    for i in range(m-1, -1, -1):
        rank = i + 1
        val = (m / rank) * ranked[i]
        prev = min(prev, val)
        q[i] = prev
    qvals = np.empty(m, dtype=float)
    qvals[order] = q
    return qvals.tolist()

# =========================
# Original corpus fraction
# =========================
def compute_original_fraction(
    bow_counts: Counter,
    ratio_master: dict,
) -> np.ndarray:
    """
    Compute per-token color-usage fractions directly from the full corpus
    (without bootstrap). This is the point estimate for each poet.
    """
    S = compute_color_scores_from_bow_ratio(bow_counts, ratio_master)
    N = sum(bow_counts.values())
    if N == 0:
        raise ValueError("BOW has zero tokens when computing original fraction.")
    return S / N  # K-dimensional vector

# =========================
# BOW bootstrap - Ratio Mode
# =========================
def bootstrap_bow_ratio(
    bow_counts: Counter,
    ratio_master: dict,
    B: int = 10000,
    seed: int = 123,
    normalize: str = "per_token",
    equalize_to: Optional[int] = None
):
    """
    Bootstrap procedure using computed ratios.

    Returns:
      - summary_colors: mean/std/CI of bootstrap distribution (for reference)
      - boots_colors:  bootstrap per-token fractions (B × K)

    Note:
      - The bootstrap samples represent the distribution of per-token fractions.
      - The original fraction estimate is computed separately (see compute_original_fraction).
    """
    if normalize not in ("none", "per_token"):
        raise ValueError('normalize must be "none" or "per_token"')

    rng = default_rng(seed)
    words = np.array(list(bow_counts.keys()))
    freqs = np.array([bow_counts[w] for w in words], dtype=int)
    N_true = int(freqs.sum())
    if N_true == 0:
        raise ValueError("BOW has zero tokens.")

    N = int(equalize_to) if (equalize_to is not None) else N_true
    if N <= 0:
        raise ValueError("Sample size N must be positive.")

    p = freqs / freqs.sum()

    boots_colors = np.empty((B, K), dtype=float)

    for b in range(B):
        idx = rng.choice(len(words), size=N, replace=True, p=p)
        sample_counts = Counter(words[idx])

        S = compute_color_scores_from_bow_ratio(sample_counts, ratio_master)
        S_out = S / N if normalize == "per_token" else S
        boots_colors[b] = S_out

    # Bootstrap summary (for reference)
    mean = boots_colors.mean(axis=0)
    std = boots_colors.std(axis=0, ddof=1)
    ci_low = np.percentile(boots_colors, 2.5, axis=0)
    ci_high = np.percentile(boots_colors, 97.5, axis=0)

    summary_colors = pd.DataFrame({
        "color": COLORS,
        "mean": mean,
        "std": std,
        "ci_low": ci_low,
        "ci_high": ci_high
    })
    return summary_colors, boots_colors

# =========================
# Per-color difference (A - B) with SE
# =========================
def diff_from_boots(bootsA: np.ndarray, bootsB: np.ndarray):
    """
    Compute per-color differences from bootstrap samples:

      - delta_mean: bootstrap mean difference (later overwritten by original Δfraction)
      - SE(delta)
      - 95% CI
      - two-sided p-value (sign test)
      - BH q-value
        
    Returns:
      diff_df, diff_samples
    """
    B = min(len(bootsA), len(bootsB))
    A = bootsA[:B]
    Bv = bootsB[:B]
    diff = A - Bv  # shape: (B × K)

    delta_mean = diff.mean(axis=0)
    ci_low = np.percentile(diff, 2.5, axis=0)
    ci_high = np.percentile(diff, 97.5, axis=0)
    se_delta = diff.std(axis=0, ddof=1)

    pvals = []
    for j in range(len(COLORS)):
        greater = (diff[:, j] >= 0).mean()
        smaller = (diff[:, j] <= 0).mean()
        p = 2 * min(greater, smaller)
        pvals.append(min(1.0, p))
    qvals = benjamini_hochberg(pvals)

    out = pd.DataFrame({
        "color": COLORS,
        "delta_mean": delta_mean,
        "se_delta": se_delta,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": pvals,
        "q_value": qvals
    })
    return out, diff

def filter_bow_by_ratio(
    bow_counts: Counter,
    ratio_master: dict,
    label: str = "AL",
    max_show: int = 20,
) -> Counter:
    """
    Filter the BOW to keep only words that exist in ratio_master.
    Prints some debug info for missing words (highest token counts).
    """
    filtered = Counter()
    missing = []

    for w, cnt in bow_counts.items():
        if w in ratio_master:
            filtered[w] = cnt
        else:
            missing.append((w, cnt))

    if missing:
        total_missing_tokens = sum(cnt for _, cnt in missing)
        print(f"\n[{label}] Words missing ratio: {len(missing)}, "
              f"Total missing tokens: {total_missing_tokens}")
        print(f"[{label}] Top {max_show} missing examples (by token count):")
        missing_sorted = sorted(missing, key=lambda x: -x[1])
        for w, cnt in missing_sorted[:max_show]:
            print(f"  {w!r}\t(count={cnt})")
    else:
        print(f"\n[{label}] All words have ratios.")

    return filtered

def check_chrom_achrom_sums(
    fractions: np.ndarray,
    label: str = "AL"
):
    """
    Verify that chromatic (7 colors) and achromatic (3 colors) fractions
    each sum close to ~1.
    """
    chrom_indices = [COLORS.index(c) for c in CHROM_COLORS]
    achrom_indices = [COLORS.index(c) for c in ACHROM_COLORS]

    chrom_sum = float(fractions[chrom_indices].sum())
    achrom_sum = float(fractions[achrom_indices].sum())

    print(f"\n[{label}] Chromatic sum = {chrom_sum:.6f} (target: ~1)")
    print(f"[{label}] Achromatic sum = {achrom_sum:.6f} (target: ~1)")

    if abs(chrom_sum - 1) > 0.05:
        print(f"  ⚠️ Chromatic sum deviates from 1 by {chrom_sum - 1:+.4f}")
    if abs(achrom_sum - 1) > 0.05:
        print(f"  ⚠️ Achromatic sum deviates from 1 by {achrom_sum - 1:+.4f}")

    return {
        "chrom_sum": chrom_sum,
        "achrom_sum": achrom_sum
    }

# =========================
# Main pipeline
# =========================

CHROM_COLORS = ["red", "orange", "yellow", "green", "blue", "purple", "pink"]
ACHROM_COLORS = ["black", "grey", "white"]

def run(
    B: int = 10000,
    seed: int = 123,
    equalize_tokens: bool = False,
):
    """
    Main execution pipeline:
      - Load precomputed ratio files
      - Build ratio master (AL + GDJ)
      - Build BOW for each poet
      - Compute original corpus fraction once (point estimate)
      - Bootstrap for SE estimation
      - Compare colors (Δfraction, p, q)
      - Save all outputs to CSV under RESULTS_PATH
    """
    print("Loading inputs...")
    al = pd.read_csv(AL_FILE)
    gdj = pd.read_csv(GDJ_FILE)

    if "content_words" not in al.columns or "raw_ratio_content_words" not in al.columns:
        raise ValueError("AL_FILE must contain 'content_words' and 'raw_ratio_content_words'.")
    if "content_words" not in gdj.columns or "raw_ratio_content_words" not in gdj.columns:
        raise ValueError("GDJ_FILE must contain 'content_words' and 'raw_ratio_content_words'.")

    al["content_words"] = al["content_words"].apply(safe_eval)
    gdj["content_words"] = gdj["content_words"].apply(safe_eval)

    al["raw_ratio_content_words"] = al["raw_ratio_content_words"].apply(safe_eval)
    gdj["raw_ratio_content_words"] = gdj["raw_ratio_content_words"].apply(safe_eval)

    # 1. Build ratio dictionaries for each poet (word → color → ratio)
    ratio_master_al = merge_ratio_dicts(al["raw_ratio_content_words"])
    ratio_master_gdj = merge_ratio_dicts(gdj["raw_ratio_content_words"])

    # 2. Merge both poets' ratio dictionaries into a unified ratio_master
    ratio_master = {}
    keys = set(ratio_master_al) | set(ratio_master_gdj)
    for w in keys:
        cmap = defaultdict(list)
        if w in ratio_master_al:
            for c, r in ratio_master_al[w].items():
                cmap[c].append(r)
        if w in ratio_master_gdj:
            for c, r in ratio_master_gdj[w].items():
                cmap[c].append(r)
        ratio_master[w] = {c: float(np.mean(v)) for c, v in cmap.items()}

    # 3. Build raw BOW for each poet
    al_bow_raw = build_bow(al)
    gdj_bow_raw = build_bow(gdj)

    # 3-1. Filter BOWs to include only words with ratio entries
    al_bow = filter_bow_by_ratio(al_bow_raw, ratio_master, label="AL")
    gdj_bow = filter_bow_by_ratio(gdj_bow_raw, ratio_master, label="GJ")

    # 4. Compute original fraction once (point estimate)
    al_frac_orig = compute_original_fraction(al_bow, ratio_master)
    gdj_frac_orig = compute_original_fraction(gdj_bow, ratio_master)
    delta_frac_orig = al_frac_orig - gdj_frac_orig

    # 4. Validate chromatic/achromatic sums
    print("\n=== Chromatic / Achromatic Sum Check ===")
    check_chrom_achrom_sums(al_frac_orig, label="AL")
    check_chrom_achrom_sums(gdj_frac_orig, label="GJ")

    # 5. Token equalization option for bootstrap
    equal_to = None
    if equalize_tokens:
        N_al = sum(al_bow.values())
        N_gdj = sum(gdj_bow.values())
        equal_to = min(N_al, N_gdj)
        print(f"Equalizing sample size to: {equal_to}")

    # 6. Bootstrap for SE estimation
    al_sum, al_boots = bootstrap_bow_ratio(
        al_bow, ratio_master, B=B, seed=seed,
        normalize="per_token", equalize_to=equal_to,
    )
    gdj_sum, gdj_boots = bootstrap_bow_ratio(
        gdj_bow, ratio_master, B=B, seed=seed+1,
        normalize="per_token", equalize_to=equal_to,
    )

    # 7. Compute bootstrap Δfraction + significance
    diff_df, diff_samples = diff_from_boots(al_boots, gdj_boots)

    # Replace bootstrap mean with original Δfraction as the final estimate
    diff_df["delta_mean"] = delta_frac_orig

    # Save per-color difference table
    mode_tag, pol_tag = "per_token_ratio", "raw_ratio"
    diff_path = os.path.join(RESULTS_PATH, f"lowell_vs_johnson_bow_diff_{mode_tag}_{pol_tag}.csv")
    diff_df.to_csv(diff_path, index=False)
    print(f"Per-color difference CSV saved to: {diff_path}")

    # 8. Summary table for reporting:
    #    - Original fraction (point estimate)
    #    - Bootstrap SE for each poet
    #    - Original Δfraction
    #    - Bootstrap SE of Δfraction
    al_se = al_boots.std(axis=0, ddof=1)
    gdj_se = gdj_boots.std(axis=0, ddof=1)
    delta_se = diff_df["se_delta"].values

    summary_df = pd.DataFrame({
        "color": COLORS,
        "al_fraction":        al_frac_orig,
        "al_fraction_se":     al_se,
        "gdj_fraction":       gdj_frac_orig,
        "gdj_fraction_se":    gdj_se,
        "delta_fraction":     delta_frac_orig,
        "delta_fraction_se":  delta_se,
        "q_value":            diff_df["q_value"].values
    })

    summary_path = os.path.join(RESULTS_PATH, "lowell_vs_johnson_color_fraction_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"Color fraction summary saved to: {summary_path}")

    print("Done. All CSVs saved to:", RESULTS_PATH)
    return {
        "al_colors_bootstrap_summary": al_sum,
        "gdj_colors_bootstrap_summary": gdj_sum,
        "diff_colors": diff_df,
        "summary": summary_df,
        "al_fraction_original": al_frac_orig,
        "gdj_fraction_original": gdj_frac_orig,
        "delta_fraction_original": delta_frac_orig,
    }

# =========================
# Execution Example
# =========================
if __name__ == '__main__':
    try:
        results = run(B=10000, seed=123, equalize_tokens=False)
        print("\nOriginal AL fractions:\n", results['al_fraction_original'])
        print("\nOriginal GDJ fractions:\n", results['gdj_fraction_original'])
        print("\nOriginal Δfraction (AL - GJ):\n", results['delta_fraction_original'])
        print("\nSummary DF head:\n", results['summary'].head())
    except FileNotFoundError as e:
        print(f"\nError: Input file not found. Please check BASE_PATH and file names: {e.filename}")
    except ValueError as e:
        print(f"\nError: Data integrity issue: {e}")
    except Exception as e:
        print(f"\nAn unexpected error occurred: {e}")


In [ ]:
# -*- coding: utf-8 -*-
"""
Visualize AL − GJ fraction difference per base color ± (CI_MULT * SE)
- Input: diff/summary CSV (containing delta_mean & SE information)
- Output: Bar plot of Δfraction per color with error bars
"""

import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

# ---- Settings ----
BASE_COLORS = ["red","orange","yellow","green","blue","purple","pink","black","grey","white"]
COLOR_MAP = {
    "red":"red","orange":"orange","yellow":"yellow","green":"green",
    "blue":"blue","purple":"purple","pink":"pink","black":"black",
    "grey":"gray","white":"white"
}

# =========================
# Relative paths (UPDATED)
# =========================
BASE_DIR = Path("./csv")

DIFF_PATH = BASE_DIR / "lowell_vs_johnson_color_fraction_summary.csv"
FIG_PATH  = BASE_DIR / "fraction_diff_errorbars.png"
SUM_PATH  = BASE_DIR / "fraction_diff_errorbars_summary.csv"

CI_MULT = 1.0     # ±1 · SE

# ---- Load & prepare data ----
df = pd.read_csv(DIFF_PATH)
df.columns = [c.strip().lower() for c in df.columns]

delta_col = "delta_fraction"
se_col = "delta_fraction_se"

# Find color column
if "color" in df.columns:
    color_col = "color"
elif "base_color" in df.columns:
    color_col = "base_color"
else:
    raise ValueError("Could not find 'color' or 'base_color' column.")

d = pd.DataFrame({
    "base_color": df[color_col].astype(str).str.lower(),
    "delta":      df[delta_col],
    "se":         df[se_col],
})

d = d[d["base_color"].isin(BASE_COLORS)].copy()
d["base_color"] = pd.Categorical(d["base_color"], categories=BASE_COLORS, ordered=True)
d.sort_values("base_color", inplace=True)

# Compute error bar
d["err_low"]  = d["delta"] - CI_MULT * d["se"]
d["err_high"] = d["delta"] + CI_MULT * d["se"]

# Save summary CSV
d.to_csv(SUM_PATH, index=False)
print(f"[Saved summary CSV] {SUM_PATH}")

# ---- Plot ----
x = np.arange(len(BASE_COLORS))
bar_w = 0.7

fig, ax = plt.subplots(figsize=(10, 8))

for i, bc in enumerate(BASE_COLORS):
    row = d[d["base_color"] == bc]
    delta = float(row["delta"])
    lo = float(row["err_low"])
    hi = float(row["err_high"])

    # Bar (height = original Δfraction)
    ax.bar(
        x[i],
        delta,
        width=bar_w,
        color=COLOR_MAP[bc],
        edgecolor="black",
        linewidth=0.8,
    )

    # Error bar
    line_color_main = "gray" if bc == "black" else "black"
    line_color_out  = "white"

    INNER_FRAC = 0.15
    OUTER_FRAC = 0.15
    half_in  = bar_w * (INNER_FRAC / 2)
    half_out = bar_w * OUTER_FRAC
    x_left   = x[i] - half_in - half_out
    x_right  = x[i] + half_in + half_out

    # Horizontal caps
    for y in (lo, hi):
        ax.hlines(y, x_left, x_right,
                  colors=line_color_out, linewidth=1.2, zorder=6)
        ax.hlines(y, x_left, x_right,
                  colors=line_color_main, linewidth=1.2, zorder=7)

    # Vertical line
    ax.vlines(
        x[i], lo, hi,
        colors=line_color_out, linewidth=1.2, zorder=6
    )
    ax.vlines(
        x[i], lo, hi,
        colors=line_color_main, linewidth=1.2, zorder=7
    )

# ---- Axis settings ----
delta_vals = d["delta"].to_numpy()
y_min, y_max = float(np.nanmin(delta_vals)), float(np.nanmax(delta_vals))
pad = max(abs(y_min), abs(y_max)) * 0.25
ax.set_ylim(y_min - pad, y_max + pad)

ax.axhline(0, color="black", lw=1)
ax.tick_params(axis='y', which='both', length=0)

ax.set_xticks(x)
ax.set_xticklabels(BASE_COLORS, fontsize=15)
ax.tick_params(axis='x', labelsize=15)

ax.set_ylabel("Difference in color usage fraction (AL − GJ) ± SE", fontsize=16)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_visible(False)
ax.spines["left"].set_visible(True)

plt.tight_layout()
plt.savefig(FIG_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"[Saved figure] {FIG_PATH}")
